In [ ]:
#Delete collection with _v2
# False: Delete a specific collection
import os
import re
from typing import List
from qdrant_client import QdrantClient

QDRANT_URL_DEFAULT = "http://100.65.71.50:6333/"

def delete_collections_suffix_v2(
    qdrant_url: str = QDRANT_URL_DEFAULT,
    api_key: str | None = None,
    dry_run: bool = True,
) -> List[str]:
    """
    Xóa mọi collection có tên kết thúc bằng 'v2' (vd: htlt_11_general_v2).
    - dry_run=True: chỉ in ra danh sách sẽ xóa, KHÔNG xóa.
    - dry_run=False: xóa thật.

    Returns: list tên collection match điều kiện.
    """
    qdrant_url = (os.getenv("QDRANT_URL", qdrant_url) or qdrant_url).rstrip("/")
    api_key = os.getenv("QDRANT_API_KEY", api_key)

    client = QdrantClient(url=qdrant_url, api_key=api_key, timeout=120)

    cols_resp = client.get_collections()
    names = [c.name for c in cols_resp.collections]

    # match suffix v2 ở cuối tên collection
    # chấp nhận: *_v2, *-v2, *.v2, hoặc ...v2 (đứng cuối)
    pat = re.compile(r"(?:[_\-.])?v2$", re.IGNORECASE)

    to_delete = [n for n in names if pat.search(n)]

    if not to_delete:
        print("No collections ending with v2 found.")
        return []

    print(f"Found {len(to_delete)} collections ending with v2:")
    for n in to_delete:
        print(" -", n)

    if dry_run:
        print("\n[dry_run=True] Not deleting anything. Set dry_run=False to delete.")
        return to_delete

    # delete thật
    for n in to_delete:
        client.delete_collection(collection_name=n)
        print(f"[DELETED] {n}")

    print(f"\nDone. Deleted {len(to_delete)} collections.")
    return to_delete
delete_collections_suffix_v2(dry_run=False)


No collections ending with v2 found.


[]

In [ ]:
#Delete one collection
qdrant_url = "http://100.65.71.50:6333/"
client = QdrantClient(url=qdrant_url, timeout=120)
client.delete_collection(collection_name="htlt_9_general_v2")

NameError: name 'QdrantClient' is not defined

In [ ]:
import os, json
from pathlib import Path
from typing import Optional, Tuple

import pandas as pd
from qdrant_client import QdrantClient


def check_qdrant_upload_vs_json(
    input_root: str,
    suffix: str,
    *,
    qdrant_url: Optional[str] = None,
    qdrant_api_key: Optional[str] = None,
    timeout: int = 120,
    only_suffix_files: bool = True,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    So sánh số lượng items trong các file JSON với số point trong Qdrant collection cùng tên (stem).
    - input_root: folder chứa các json (recursive)
    - suffix: hậu tố để lọc, ví dụ 'v3' (case-insensitive)
    - only_suffix_files: True -> chỉ check các file có stem endswith suffix (đúng nhu cầu bạn)
    Trả về:
      - df_checked: bảng các file đã check (thường là v3)
      - df_bad: subset mismatch / missing / bad json
    """
    qdrant_url = (qdrant_url or os.getenv("QDRANT_URL", "http://100.65.71.50:6333/")).rstrip("/")
    qdrant_api_key = qdrant_api_key if qdrant_api_key is not None else os.getenv("QDRANT_API_KEY", None)

    # CONNECT
    client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key, timeout=timeout)

    # LIST COLLECTIONS (endswith suffix)
    collections = client.get_collections().collections
    col_names = sorted([c.name for c in collections if c.name.lower().endswith(suffix.lower())])
    col_set = set(col_names)

    root = Path(input_root).resolve()
    json_files = sorted([p for p in root.rglob("*.json") if p.is_file() and not p.name.startswith("_")])

    if verbose:
        print(f"Qdrant URL: {qdrant_url}")
        print(f"Found collections ending with '{suffix}': {len(col_names)}")
        print(f"Input root: {root}")
        print(f"Found json files under input_root: {len(json_files)}")

    rows = []
    for p in json_files:
        collection_name = p.stem
        want_check = collection_name.lower().endswith(suffix.lower())

        if only_suffix_files and (not want_check):
            continue

        try:
            data = json.loads(p.read_text(encoding="utf-8"))
            json_is_list = isinstance(data, list)
            json_count = len(data) if json_is_list else None
        except Exception:
            json_is_list = False
            json_count = None

        qdrant_exists = collection_name in col_set
        qdrant_count = None
        qdrant_status = "missing"
        if qdrant_exists:
            try:
                cnt = client.count(collection_name=collection_name, exact=True)
                qdrant_count = int(cnt.count)
                qdrant_status = "ok"
            except Exception as e:
                qdrant_status = f"count_error: {type(e).__name__}"

        match = (json_count is not None and qdrant_count is not None and json_count == qdrant_count)

        rows.append({
            "file": str(p.relative_to(root)),
            "stem": collection_name,
            "check_suffix": want_check,
            "json_is_list": json_is_list,
            "json_items": json_count,
            "qdrant_exists": qdrant_exists,
            "qdrant_points": qdrant_count,
            "qdrant_status": qdrant_status,
            "match": match,
        })

    df = pd.DataFrame(rows)

    # Nếu only_suffix_files=True thì df đã là v3 rồi; còn không thì lọc tiếp
    df_checked = df if only_suffix_files else df[df["check_suffix"] == True].copy()

    if verbose:
        print("\n=== SUMMARY ===")
        print("total_checked:", len(df_checked))
        print("missing_collections:", int((df_checked["qdrant_exists"] == False).sum()))
        print("mismatch_counts:", int(((df_checked["qdrant_exists"] == True) & (df_checked["match"] == False)).sum()))
        print("bad_json:", int((df_checked["json_is_list"] == False).sum()))
        print("all_ok:", bool((df_checked["qdrant_exists"] == True).all() and (df_checked["match"] == True).all() and (df_checked["json_is_list"] == True).all()))

    df_bad = df_checked[
        (df_checked["json_is_list"] == False) |
        (df_checked["qdrant_exists"] == False) |
        (df_checked["match"] == False)
    ].copy()

    return df_checked, df_bad

df_v3, bad = check_qdrant_upload_vs_json(
    input_root=r"E:\QuangNV\Chunking_Final\z\chunking_super_hybrid\outputs\chunking_08012026_v0\04_dataset_09012026",
    suffix="v3",
    verbose=True,
)

display(bad.sort_values(["qdrant_exists", "match", "stem"])[
    ["stem", "json_items", "qdrant_points", "qdrant_exists", "qdrant_status", "file"]
])

Qdrant URL: http://100.65.71.50:6333
Found collections ending with 'v2': 60
Input root: E:\QuangNV\Chunking_Final\z\chunking_super_hybrid\outputs\final_06012026\04_dataset_07012026
Found json files under input_root: 60

=== SUMMARY ===
total_checked: 60
missing_collections: 0
mismatch_counts: 0
bad_json: 0
all_ok: True


,stem,json_items,qdrant_points,qdrant_exists,qdrant_status,file
